# Cantilever Beam — 3D Linear Elasticity

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/camlab-ethz/TensorMesh/blob/main/notebooks/cantilever_beam.ipynb)

A steel beam clamped at one end and loaded at the other: the entry point to
solid mechanics in TensorMesh, and the first notebook where the unknown is a
**vector** field. `LinearElasticityElementAssembler` produces one $3\times3$
block per node pair, so the global system is $[3N, 3N]$ — but the calling code
is the same assemble → condense → solve as everywhere else.

The tip deflection is checked against Euler-Bernoulli beam theory, and the
deformed shape is rendered with pyvista.

🖥️ *Installs pyvista and a virtual framebuffer, so the first cell takes a
little longer than in the 2D notebooks.*

Docs: [Cantilever Beam](https://docs.tensor-mesh.com/example_gallery/solid/cantilever_beam.html) · Source: [`examples/solid/cantilever_beam/cantilever_beam.py`](https://github.com/camlab-ethz/TensorMesh/blob/main/examples/solid/cantilever_beam/cantilever_beam.py)

In [ ]:
# Install TensorMesh (skipped automatically if it is already available, e.g. a local dev setup).
# The apt line provides the OpenGL utility library that gmsh -- TensorMesh's mesh generator --
# needs at import time; it is a no-op where the library is already present.
# pyvista renders the 3D figures; xvfb gives it a virtual display to draw into.
import importlib.util
if importlib.util.find_spec("tensormesh") is None:
    !apt-get -qq install -y libglu1-mesa libgl1-mesa-glx xvfb > /dev/null 2>&1 || true
    %pip install -q tensormesh-fem==0.2.0 pyvista

import contextlib
import os


@contextlib.contextmanager
def quiet():
    """Hide gmsh's meshing log, which is written below Python's stdout.
    Drop the ``with quiet():`` wrapper anywhere to see what the mesher is doing."""
    with open(os.devnull, "w") as null:
        saved = os.dup(1)
        os.dup2(null.fileno(), 1)
        try:
            yield
        finally:
            os.dup2(saved, 1)
            os.close(saved)

In [ ]:
import warnings

import pyvista as pv
import vtk

vtk.vtkObject.GlobalWarningDisplayOff()          # software-OpenGL fallback notices
warnings.filterwarnings("ignore", message=".*Use vtk with osmesa.*")
warnings.filterwarnings("ignore", category=DeprecationWarning, module="pyvista.*")
pv.OFF_SCREEN = True

import torch

from tensormesh import Condenser, Mesh
from tensormesh.assemble import LinearElasticityElementAssembler
from tensormesh.material import Steel
from tensormesh.visualization import plot_deformation

## Assemble and solve

`tensormesh.material` carries the usual engineering constants, so the physics
reads in SI units rather than in non-dimensional placeholders.

In [ ]:
# 1. Geometry: a 2 m x 0.2 m x 0.2 m steel beam, meshed with tetrahedra.
with quiet():
    mesh = Mesh.gen_cube(chara_length=0.08, left=0.0, right=2.0,
                         bottom=0.0, top=0.2, front=0.0, back=0.2)
n_nodes = mesh.points.shape[0]
print(f"mesh: {n_nodes} nodes, {mesh.n_elements} elements")
print(f"material: {Steel.name}  E={Steel.E / 1e9:g} GPa  nu={Steel.nu}")

# 2. Stiffness. The assembler returns one 3x3 block per node pair, so the global
#    matrix is [3N, 3N] -- displacement is a vector field, unlike every scalar
#    example so far.
K = LinearElasticityElementAssembler.from_mesh(mesh, E=Steel.E, nu=Steel.nu)()
print(f"stiffness: {K.shape}")

# 3. Clamp the x = 0 face: three DOFs per fixed node.
points = mesh.points
eps = 1e-5
fixed_node_mask = torch.abs(points[:, 0]) < eps
condenser = Condenser(torch.repeat_interleave(fixed_node_mask, 3))

# 4. Load: 100 kN downwards, spread over the free end face.
right_nodes = torch.where(torch.abs(points[:, 0] - 2.0) < eps)[0]
rhs = torch.zeros((n_nodes, mesh.dim))
rhs[right_nodes, 1] = -1.0e5 / right_nodes.shape[0]

# 5. Solve.
K_, f_ = condenser(K, rhs.flatten())
u = condenser.recover(K_.solve(f_).float()).reshape(-1, 3)

max_disp = torch.norm(u, dim=1).max().item()
print(f"max displacement: {max_disp * 1000:.2f} mm")

## Sanity check against beam theory

A 2 m beam of 0.2 m square section is on the stocky side, so the classical
slender-beam formula is a reference rather than ground truth.

In [ ]:
# Euler-Bernoulli reference for an end-loaded cantilever: delta = F L^3 / (3 E I).
L, b, h = 2.0, 0.2, 0.2
I = b * h ** 3 / 12
analytic = 1.0e5 * L ** 3 / (3 * Steel.E * I)
print(f"FEM       : {max_disp * 1000:.2f} mm")
print(f"beam theory: {analytic * 1000:.2f} mm")
print(f"ratio      : {max_disp / analytic:.3f}")
print("\nBeam theory ignores shear deformation and treats the clamp as perfectly")
print("rigid, so for a beam this stocky the two are close but not identical.")

## The deformed shape

In [ ]:
# Exaggerate the deformation so it is visible; the real deflection is millimetres
# on a two-metre beam.
scale = 0.4 / max_disp
print(f"visualisation scale factor: {scale:.0f}x")

plot_deformation(mesh, u, "cantilever_steel.png", scale_factor=scale,
                 camera_position="xy", fixed_nodes=fixed_node_mask,
                 force_vectors=rhs)

from IPython.display import Image
Image("cantilever_steel.png")

## Where to next

- [Hyperelastic beam](https://colab.research.google.com/github/camlab-ethz/TensorMesh/blob/main/notebooks/hyperelastic_beam.ipynb) — the same geometry at large deformation, solved by energy minimisation.
- [The solid gallery](https://docs.tensor-mesh.com/example_gallery/solid/index.html) — plasticity, contact, geomechanics, vibration modes.